# 예제 IV · Post-Training Quantization

사전 학습된 **CIFAR-10 ResNet-20**의 Weight와 실제 ReLU Activation에 양자화·복원을 적용합니다. 상단에서 **Weight와 Activation을 독립적으로 설정**하고, 선택한 한 조합에 대해 복원 오차와 테스트 정확도를 확인합니다.

**기본 설정**은 Weight 8-bit / Symmetric / Uniform / Per-channel, Activation 8-bit / Asymmetric / Uniform / Per-tensor입니다. 노트북은 양자화 후 복원한 값을 부동소수점으로 계산하는 **Q/DQ 시뮬레이션**입니다.

**코드 셀 1 · Weight와 Activation 양자화 설정**

Weight와 Activation의 비트 수, 범위 방식, Grid와 Granularity를 지정하고 Calibration 128장과 테스트 512장을 설정합니다.

In [ ]:
# @title 실습 설정
# Weight의 범위 방식, 눈금 간격, 변환 단위를 정한다. Group Size는 Per-group에 적용된다.
WEIGHT_BITS = 8 # @param {type:"integer"}
WEIGHT_SCHEME = "symmetric" # @param ["symmetric", "asymmetric"]
WEIGHT_GRID = "uniform" # @param ["uniform", "non-uniform"]
WEIGHT_GRANULARITY = "per-channel" # @param ["per-tensor", "per-channel", "per-group"]
WEIGHT_GROUP_SIZE = 16 # @param {type:"integer"}

# Activation은 Weight와 별도로 설정한다. Calibration에 선택한 변환 단위가 적용된다.
ACTIVATION_BITS = 8 # @param {type:"integer"}
ACTIVATION_SCHEME = "asymmetric" # @param ["symmetric", "asymmetric"]
ACTIVATION_GRID = "uniform" # @param ["uniform", "non-uniform"]
ACTIVATION_GRANULARITY = "per-tensor" # @param ["per-tensor", "per-channel", "per-group"]
ACTIVATION_GROUP_SIZE = 32 # @param {type:"integer"}

# μ는 Non-uniform에서 로그 압축의 강도를 정한다. 아래 영상 수는 수업 중 실행 시간을 조절한다.
MU = 15.0 # @param {type:"number"}
CALIBRATION_SAMPLES = 128 # @param {type:"integer"}
TEST_SAMPLES = 512 # @param {type:"integer"}

## 1. 옵션의 의미

| 설정 | 선택지 | 어떤 값이 바뀌는가 |
|---|---|---|
| 범위 | Symmetric / Asymmetric | Symmetric은 실수 0을 중심으로 좌우 범위를 같게 둡니다. Asymmetric은 관측한 음수·양수 범위를 따로 반영합니다. |
| 복원값 간격 | Uniform / Non-uniform | Uniform은 일정 간격을 사용합니다. Non-uniform은 이 실습의 μ-law companding으로 0 부근에 복원값을 촘촘히 배치합니다. |
| Granularity | Per-tensor / Per-channel / Per-group | 텐서 전체, 채널별, 채널 안의 그룹별로 변환 기준을 따로 정합니다. |
| 비트 수 및 그룹 크기 | Weight·Activation 개별 설정 | 비트 수가 정수 코드의 범위를 정합니다. 그룹 크기는 Per-group에서만 사용합니다. |

**Weight**의 Per-channel은 **출력 채널별** 기준입니다. Per-group에서는 각 출력 채널의 Weight를 펼쳐 연속 원소를 묶습니다. **Activation**의 Per-group에서는 각 Feature Map 채널의 공간 위치(H×W)를 펼쳐 묶고, Calibration 영상의 같은 위치 묶음에서 범위를 관측합니다.

Non-uniform의 Asymmetric 설정은 관측 범위에 따라 0 아래와 위에 사용할 코드 수를 나눕니다. [μ-law 함수 참고](https://docs.pytorch.org/audio/stable/generated/torchaudio.transforms.MuLawEncoding.html)

**코드 셀 2 · 실행 장치와 캐시 설정**

PyTorch와 torchvision을 불러오고 실습에 사용할 GPU 또는 CPU, 시드, 데이터와 가중치 저장 경로를 설정합니다.

In [ ]:
# 실습 설정을 한곳에서 관리한다. 데이터·가중치 캐시를 재사용하고 CUDA가 있으면 GPU로 실행한다.
import copy
import os
import time
from itertools import islice
from pathlib import Path
from statistics import mean

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# 동일한 시드와 배치 수를 사용해 반복 실행 조건을 맞춘다.
SEED = 50
BATCH_SIZE = 64
DATA_ROOT = Path(os.environ.get("AX_IDEC_DATA_ROOT", "./data"))
ASSET_ROOT = Path(os.environ.get("AX_IDEC_CACHE", "./idec_cache"))
ASSET_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
print("실행 장치:", torch.cuda.get_device_name() if DEVICE.type == "cuda" else "CPU")

## 2. 사전 학습 모델과 데이터 준비

사전 학습 Weight를 불러와 ResNet-20을 평가 모드로 둡니다. 코드의 두 Loader는 역할이 다릅니다.

- **Calibration 128장** · 학습 분할에서 뽑은 대표 영상. Activation 범위를 정하는 데 사용
- **테스트 512장** · 테스트 분할에서 뽑은 고정 영상. FP32와 선택한 Q/DQ 모델의 정확도 비교에 사용

데이터와 체크포인트가 캐시에 있으면 다음 실행에서 재사용합니다.

**코드 셀 3 · 모델 다운로드 함수 정의**

고정된 GitHub 커밋의 소스와 가중치를 내려받아 캐시에 저장하고 ZIP의 항목 경로를 확인한 뒤 압축을 푸는 함수를 정의합니다.

In [ ]:
from urllib.request import urlretrieve
from zipfile import ZipFile
import sys

# 체크포인트와 소스를 캐시에 보관한다. 다운로드가 완료되면 임시 파일을 정상 파일 이름으로 바꾼다.
def fetch(url, filename):
    destination = ASSET_ROOT / filename
    if not destination.is_file():
        pending = destination.with_suffix(destination.suffix + ".download")
        try:
            urlretrieve(url, pending)
            pending.replace(destination)
        finally:
            pending.unlink(missing_ok=True)
    return destination

# 고정한 커밋의 ZIP을 받고, 각 항목의 경로를 검사한 뒤 추출한다.
def fetch_repository(owner, repository, revision, filename):
    archive = fetch(f"https://codeload.github.com/{owner}/{repository}/zip/{revision}", filename)
    root = ASSET_ROOT / f"{repository}-{revision}"
    if not root.exists():
        with ZipFile(archive) as source_zip:
            for entry in source_zip.infolist():
                destination = (ASSET_ROOT / entry.filename).resolve()
                if not destination.is_relative_to(root.resolve()):
                    raise ValueError(f"Unexpected archive path: {entry.filename}")
            source_zip.extractall(ASSET_ROOT)
    return root.resolve()

**코드 셀 4 · ResNet-20과 Calibration 데이터 준비**

CIFAR-10으로 사전 학습한 ResNet-20을 불러오고 학습 분할의 Calibration 영상과 별도 테스트 분할의 평가 영상을 준비합니다.

In [ ]:
import matplotlib.pyplot as plt

# 고정한 소스의 ResNet-20 구조에 공개 CIFAR-10 학습 가중치를 로드한다.
RESNET20_REVISION = "786c16252c0fc58ee9adac063f8337cc4a7a497a"
resnet_source = fetch_repository("chenyaofo", "pytorch-cifar-models", RESNET20_REVISION, "resnet-cifar.zip")
sys.path.insert(0, str(resnet_source))
from pytorch_cifar_models import cifar10_resnet20

model = cifar10_resnet20(pretrained=False)
weights_url = ("https://github.com/chenyaofo/pytorch-cifar-models/releases/download/"
               "resnet/cifar10_resnet20-4118986f.pt")
weights_path = fetch(weights_url, "resnet20.pt")
model.load_state_dict(torch.load(weights_path, map_location="cpu", weights_only=True))
model.to(DEVICE).eval()

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])
# 학습 분할의 대표 영상으로 Calibration 범위를 정하고, 별도 테스트 분할에서 정확도를 비교한다.
calibration_data = datasets.CIFAR10(DATA_ROOT, train=True, download=True, transform=transform)
test_data = datasets.CIFAR10(DATA_ROOT, train=False, download=True, transform=transform)
indices = torch.randperm(len(calibration_data), generator=torch.Generator().manual_seed(SEED))
eval_indices = torch.randperm(len(test_data), generator=torch.Generator().manual_seed(SEED + 1))
calibration_loader = DataLoader(Subset(calibration_data, indices[:CALIBRATION_SAMPLES].tolist()), batch_size=BATCH_SIZE)
test_loader = DataLoader(Subset(test_data, eval_indices[:TEST_SAMPLES].tolist()), batch_size=BATCH_SIZE)
print(f"Calibration {len(calibration_loader.dataset)}장 / 테스트 {len(test_loader.dataset)}장")

## 3. 그룹별 범위와 양자화·복원 함수

1. `tensor_blocks()`가 변환 기준을 공유할 값들을 한 행으로 모읍니다. 변환 후에는 다시 원래 Tensor 모양으로 돌려놓습니다.
2. `quantization_parameters()`가 비트 수와 관측 범위로부터 정수 코드 범위와 Zero-point를 정합니다. Uniform에서는 Scale이 실수의 눈금 간격입니다.
3. `quantize_dequantize()`는 선택한 눈금에 값을 대응시키고 실수값으로 복원합니다. Non-uniform에서는 로그 압축과 그 역함수를 사용합니다.

Weight의 범위는 Weight 자체에서 계산합니다. Activation의 범위는 다음 단계에서 관측해 고정합니다.

**코드 셀 5 · 양자화와 복원 함수 정의**

Weight와 Activation의 설정을 검증하고, 선택한 범위 방식과 Grid 및 Granularity에 따라 값을 양자화한 뒤 실수로 복원하는 Q/DQ 함수를 정의합니다.

In [ ]:
from dataclasses import dataclass
import math

# 설정값을 하나로 묶고 비트 수와 선택지가 지원 범위를 벗어나지 않았는지 확인한다.
@dataclass(frozen=True)
class QuantConfig:
    bits: int
    scheme: str
    grid: str
    granularity: str
    group_size: int
    mu: float = 15.0

    def __post_init__(self):
        if not 2 <= self.bits <= 8:
            raise ValueError("비트 수는 2~8 사이로 설정하세요.")
        if self.scheme not in {"symmetric", "asymmetric"}:
            raise ValueError(f"지원하지 않는 scheme: {self.scheme}")
        if self.grid not in {"uniform", "non-uniform"}:
            raise ValueError(f"지원하지 않는 grid: {self.grid}")
        if self.granularity not in {"per-tensor", "per-channel", "per-group"}:
            raise ValueError(f"지원하지 않는 granularity: {self.granularity}")
        if self.group_size < 1 or not math.isfinite(self.mu) or self.mu <= 0:
            raise ValueError("Group size와 μ는 양수여야 합니다.")

weight_cfg = QuantConfig(WEIGHT_BITS, WEIGHT_SCHEME, WEIGHT_GRID,
                         WEIGHT_GRANULARITY, WEIGHT_GROUP_SIZE, MU)
activation_cfg = QuantConfig(ACTIVATION_BITS, ACTIVATION_SCHEME, ACTIVATION_GRID,
                             ACTIVATION_GRANULARITY, ACTIVATION_GROUP_SIZE, MU)

# Scale과 Zero-point를 함께 사용할 값들을 행으로 만든다. 변환 후 원래 축으로 되돌리는 함수도 반환한다.
def tensor_blocks(x, config, activation=False):
    # 블록 행마다 하나의 독립적인 변환 기준을 적용합니다.
    if config.granularity == "per-tensor":
        return x.reshape(1, -1), lambda blocks: blocks.reshape_as(x)

    rows = x.movedim(1, 0) if activation else x
    channels = rows.shape[0]
    # Per-channel에서는 한 채널마다 별도 변환 기준을 쓴다. Activation의 채널 축은 앞에 옮겨 둔다.
    if config.granularity == "per-channel":
        def restore(blocks):
            output = blocks.reshape_as(rows)
            return output.movedim(0, 1) if activation else output
        return rows.reshape(channels, -1), restore

    if activation:
        samples = x.shape[0]
        rows_by_sample = rows.reshape(channels, samples, -1)
    else:
        samples = 1
        rows_by_sample = rows.reshape(channels, 1, -1)

    length = rows_by_sample.shape[-1]
    group_size = config.group_size
    groups = (length + group_size - 1) // group_size
    # 범위는 항상 0을 포함하도록 정하므로 끝 그룹의 0 padding은 기준을 바꾸지 않습니다.
    # Per-group에서도 마지막 원소를 포함하도록 0을 채우고, 같은 공간 그룹을 영상끼리 묶는다.
    padded = F.pad(rows_by_sample, (0, groups * group_size - length))
    blocks = padded.reshape(channels, samples, groups, group_size)
    blocks = blocks.permute(0, 2, 1, 3).reshape(channels * groups, samples * group_size)

    def restore(quantized):
        result = quantized.reshape(channels, groups, samples, group_size)
        result = result.permute(0, 2, 1, 3).reshape(channels, samples, -1)
        result = result[..., :length].reshape_as(rows)
        return result.movedim(0, 1) if activation else result

    return blocks, restore

# 관측 범위에 0을 포함시킨다. Symmetric은 절댓값 최댓값 기준, Asymmetric은 전체 범위 폭과 Zero-
# point 기준이다.
def quantization_parameters(low, high, config):
    low = low.clamp_max(0)
    high = high.clamp_min(0)
    if config.scheme == "symmetric":
        qmax = 2 ** (config.bits - 1) - 1
        peak = torch.maximum(-low, high)
        scale = torch.where(peak > 0, peak / qmax, 1.0)
        zero = torch.zeros_like(scale)
        return -peak, peak, scale, zero, -qmax, qmax

    qmax = 2**config.bits - 1
    width = high - low
    scale = torch.where(width > 0, width / qmax, 1.0)
    zero = (-low / scale).round().clamp(0, qmax)
    return low, high, scale, zero, 0, qmax

# Weight는 현재 범위에서, Activation은 Calibration에서 고정한 범위로 양자화·복원한다.
def quantize_dequantize(x, config, activation=False, bounds=None):
    blocks, restore = tensor_blocks(x, config, activation)
    low, high = blocks.aminmax(dim=1, keepdim=True) if bounds is None else bounds
    lo, hi, scale, zero, qmin, qmax = quantization_parameters(low, high, config)
    if config.grid == "uniform":
        codes = (blocks / scale).round().add(zero).clamp(qmin, qmax)
        return restore((codes - zero) * scale)

    # μ-law companding: 음수·양수 영역을 각자의 끝값으로 정규화한 뒤 복원합니다.
    # Non-uniform은 양·음 영역을 0 기준으로 나눠 μ-law로 압축한다. 복원하면 0 근처 눈금이 촘촘해진다.
    negative = blocks < 0
    span = torch.where(negative, -lo, hi)
    levels = torch.where(negative, zero - qmin, qmax - zero)
    safe_span = torch.where(span > 0, span, 1.0)
    normalized = blocks.abs().div(safe_span).clamp(0, 1)
    normalized = torch.where(span > 0, normalized, 0.0)
    log_mu = math.log1p(config.mu)
    compressed = torch.log1p(config.mu * normalized) / log_mu
    # 압축 영역에서 정수 단계로 반올림한 뒤 역함수로 실수 크기를 복원한다.
    code_steps = (compressed * levels).round()
    fraction = code_steps / levels.clamp_min(1)
    reconstructed = torch.expm1(fraction * log_mu) * (span / config.mu)
    return restore(torch.where(negative, -reconstructed, reconstructed))

## 4. Activation Calibration

대표 영상을 FP32 ResNet-20에 입력하고 ReLU 출력이 관측될 때마다 Min/Max를 모읍니다. 평가 때는 이 범위를 재사용합니다.

**첫 ReLU의 범위 개수**를 확인합니다. Per-tensor에서 한 개이며, Per-channel 또는 Per-group에서는 선택한 채널·그룹 수만큼 범위를 관리합니다. 이 숫자는 어떤 값들이 같은 변환 기준을 쓰는지 확인하는 단서입니다.

**코드 셀 6 · ReLU Activation 범위 측정**

Calibration 영상을 통과시키며 각 ReLU의 최솟값과 최댓값을 수집하고 첫 ReLU의 Activation 예시를 저장합니다.

In [ ]:
# ReLU Hook으로 각 배치의 범위를 수집하고, 전체 Calibration 동안의 Min/Max를 합친다.
@torch.inference_mode()
def calibrate_model(model, loader, config, capture_sample=False):
    ranges = {}
    sample = {}

    def collect(name):
        def update(_module, _inputs, output):
            blocks, _ = tensor_blocks(output, config, activation=True)
            lo, hi = blocks.aminmax(dim=1, keepdim=True)
            if name in ranges:
                previous_lo, previous_hi = ranges[name]
                lo = torch.minimum(previous_lo, lo)
                hi = torch.maximum(previous_hi, hi)
            ranges[name] = (lo, hi)
            if capture_sample and not sample:
                sample.update(name=name, activation=output[:1].detach().clone())
        return update

    # 실행 중 필요한 계층에만 Hook을 연결하고 블록이 끝나면 제거한다.
    hooks = [
        module.register_forward_hook(collect(name))
        for name, module in model.named_modules() if isinstance(module, nn.ReLU)
    ]
    try:
        model.eval()
        for images, _ in loader:
            model(images.to(DEVICE))
    finally:
        for hook in hooks:
            hook.remove()
    return ranges, sample

# 선택한 Activation 설정으로 범위를 수집하고, 분포를 그릴 실제 샘플을 보관한다.
ranges, sample = calibrate_model(model, calibration_loader, activation_cfg, capture_sample=True)
print(f"Activation 범위를 측정한 ReLU: {len(ranges)}개")
print(f"첫 ReLU의 범위 개수: {ranges[sample['name']][0].shape[0]}")

## 5. 원본과 복원값 비교

첫 Conv2d의 Weight와 첫 ReLU 출력에서 **FP32와 Q/DQ 결과의 Histogram**을 비교합니다. 평균 절대 복원 오차는 같은 원소끼리 `|원본−복원값|`을 계산해 평균 낸 값입니다.

Histogram에서는 값이 어느 구간에 얼마나 몰리는지 보고, 복원 오차는 수치로 비교합니다. 이 셀의 두 오차 값은 표시한 계층·샘플에 대한 결과입니다.

**코드 셀 7 · Weight와 Activation 복원 오차 확인**

첫 Conv2d의 Weight와 첫 ReLU의 Activation에서 FP32와 Q/DQ의 값 분포를 그려 비교하고 평균 절대 복원 오차를 출력합니다.

In [ ]:
with torch.inference_mode():
    weight = model.conv1.weight.detach()
    activation = sample["activation"]
    # 첫 Conv2d Weight와 실제 ReLU Activation의 복원값을 계산해 원본과 비교한다.
    restored_weight = quantize_dequantize(weight, weight_cfg)
    restored_activation = quantize_dequantize(
        activation, activation_cfg, activation=True, bounds=ranges[sample["name"]]
    )

weight_blocks, _ = tensor_blocks(weight, weight_cfg)
print(f"첫 Conv2d Weight의 범위 개수: {weight_blocks.shape[0]}")
# 원본과 처리한 결과를 나란히 확인할 두 개의 시각화 영역을 만든다.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True)
for axis, title, raw, restored in [
    (axes[0], "Conv1 Weight", weight, restored_weight),
    (axes[1], "ReLU Activation", activation, restored_activation),
]:
    axis.hist(raw.flatten().cpu().numpy(), bins=32, alpha=0.5, label="FP32")
    axis.hist(restored.flatten().cpu().numpy(), bins=32, alpha=0.5, label="Q/DQ")
    axis.set_title(title)
    axis.legend()
    print(f"{title} 평균 절대 복원 오차: {(raw - restored).abs().mean().item():.6f}")
plt.show()

## 6. 같은 CIFAR-10 테스트 샘플에서 평가

선택한 Weight·Activation 변환을 복제 모델에 적용한 뒤, 같은 영상에서 **FP32 정확도**, **Q/DQ 정확도**, **FP32 대비 차이(%p)**를 확인합니다.

설정의 영향을 비교할 때는 비트 수·범위·간격·Granularity 중 **한 항목만 바꾸고** 다른 항목과 테스트 샘플은 유지합니다. 설정 셀을 수정한 뒤 위에서부터 다시 실행하면 새 Activation 설정에 맞춰 Calibration도 다시 수행됩니다.

**코드 셀 8 · Q/DQ 적용과 평가 함수 정의**

복제한 모델의 Conv2d와 Linear Weight를 Q/DQ하고 평가 중에는 측정한 범위로 ReLU Activation을 Q/DQ하는 함수를 준비합니다.

In [ ]:
# 기준 모델을 복제하고 Conv2d와 Linear의 Weight에만 Q/DQ를 적용한다.
@torch.inference_mode()
def quantized_weight_model(model, config):
    modified = copy.deepcopy(model).eval()
    for module in modified.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            module.weight.copy_(quantize_dequantize(module.weight, config))
    return modified

# 평가 중에는 이미 정한 Activation 범위로 ReLU 출력에 Q/DQ를 적용한다.
def attach_activation_hooks(model, config, ranges):
    def make_hook(bounds):
        def apply(_module, _inputs, output):
            return quantize_dequantize(output, config, activation=True, bounds=bounds)
        return apply

    return [
        module.register_forward_hook(make_hook(ranges[name]))
        for name, module in model.named_modules() if name in ranges
    ]

# 평가 모드에서 각 영상의 최댓값 클래스를 정답과 비교해 전체 정확도를 집계한다.
@torch.inference_mode()
def evaluate(model, loader):
    correct, count = 0, 0
    model.eval()
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        correct += (model(images).argmax(dim=1) == labels).sum().item()
        count += labels.numel()
    return 100 * correct / count

**코드 셀 9 · FP32와 Q/DQ 정확도 비교**

FP32 모델과 Weight·Activation에 Q/DQ를 적용한 모델을 같은 테스트 512장으로 평가해 정확도와 변화 폭을 출력합니다.

In [ ]:
start = time.perf_counter()
# 같은 테스트 샘플에서 FP32 기준 모델과 Weight·Activation에 Q/DQ를 적용한 모델을 비교한다.
simulated = quantized_weight_model(model, weight_cfg)
hooks = attach_activation_hooks(simulated, activation_cfg, ranges)
try:
    before = evaluate(model, test_loader)
    after = evaluate(simulated, test_loader)
finally:
    for hook in hooks:
        hook.remove()

print("Weight 설정:", weight_cfg)
print("Activation 설정:", activation_cfg)
print(f"FP32 정확도: {before:.2f}%")
print(f"Q/DQ 정확도: {after:.2f}%")
print(f"FP32 대비 변화: {after - before:+.2f}%p")
print(f"변환 및 비교 평가: {time.perf_counter() - start:.2f} s")